# 1102 42일차

WSL 배포판 정리, 권한 트러블슈팅, SSH 원격 접속 및 Furiosa NPU 환경에서의 LLM 모델 서빙

## 1. WSL 배포판 삭제 및 전환

더 이상 쓰지 않는 WSL 배포판을 정리하고 필요한 배포판으로 전환해 접속하는 방법

WSL 배포판마다 독립된 파일 시스템(가상 디스크)을 차지하므로, 쓰지 않는 배포판을 정리해 용량을 확보하고 특정 환경으로 정확히 진입하려고 씀

1. `wsl -l -v` : 설치된 배포판 목록, 실행 상태, WSL 버전 확인
2. `wsl --unregister Ubuntu-24.04` : `Ubuntu-24.04` 배포판 완전 삭제
3. `wsl -d furiosa02` : `furiosa02` 배포판으로 지정 접속
4. `wsl -s furiosa02` : `furiosa02`를 기본(Default) 배포판으로 지정

- `--unregister`는 가상 디스크 파일(`ext4.vhdx`)까지 통째로 삭제하므로 복구가 불가능함. 필요한 파일은 미리 백업해야 함
- `-d <이름>` 옵션으로 원하는 배포판에 바로 들어갈 수 있음
- `-s <이름>`으로 기본 배포판을 지정해 두면, 이후 `wsl`만 쳐도 해당 배포판으로 바로 열림

## 2. WSL 파일 시스템 접근 및 권한 트러블슈팅

WSL 내부 디렉터리를 윈도우 탐색기로 열거나 편집할 때 발생하는 경로/권한 오류 해결

윈도우와 WSL 간 9P/virtio-fs 공유 캐시 문제나 리눅스 계정 소유권 문제로 탐색기 접근 및 파일 수정이 막히는 것을 해결하려고 씀

1. `explorer.exe $(wslpath -w "$PWD")` : 현재 리눅스 경로를 윈도우 경로로 정확히 변환해 탐색기 열기
2. `wsl --shutdown` : WSL 가상 머신 및 네트워크 공유 세션을 완전히 재시작
3. `sudo chown -R $USER:$USER .` : 현재 디렉터리의 소유권을 현재 로그인한 리눅스 사용자에게 부여
4. `sudo chmod -R 777 .` : 모든 사용자에게 읽기/쓰기/실행 권한 부여

- `explorer.exe .` 실행 시 "잘못된 주소에 접근하려고 했습니다" 오류가 뜨는 것은 `\\wsl.localhost` 경로 변환 또는 캐시가 꼬였기 때문
- 탐색기에서 파일 생성/수정 시 "컴퓨터/사용자에서 권한을 부여받아야 합니다" 오류는 해당 폴더가 리눅스 `root` 소유이거나 일반 사용자 쓰기 권한이 없을 때 발생함
- `chown -R $USER:$USER .`로 소유자를 내 계정으로 바꿔주면 윈도우 탐색기에서도 자유롭게 파일 수정이 가능해짐

## 3. SSH 키 복사 및 원격 서버 접속

새로운 WSL 환경에서 윈도우에 있던 SSH 키를 가져와 원격 Furiosa 서버에 안전하게 접속하는 방법

새로 생성한 WSL 배포판에는 기존 키가 없으므로 윈도우 C 드라이브에서 키를 복사해 오고 올바른 보안 권한을 부여해 접속함

1. `mkdir -p ~/.ssh` : 홈 디렉터리에 `.ssh` 폴더 생성
2. `cp /mnt/c/Users/Admin/.ssh/id_rsa_15 ~/.ssh/` : 윈도우에 저장된 개인키를 WSL로 복사
3. `chmod 600 ~/.ssh/id_rsa_15` : 개인키 파일 권한을 소유자 전용(읽기/쓰기)으로 설정
4. `ssh -o StrictHostKeyChecking=accept-new -i ~/.ssh/id_rsa_15 -p <포트> root@<서버 주소>` : 첫 접속 시 호스트 지문을 자동 수락하며 접속

- `Warning: Identity file .ssh/id_rsa_15 not accessible` 경고는 키 파일이 없거나 상대 경로(`.ssh/`)로 지정되어 현재 위치에서 파일을 찾지 못할 때 발생함. 항상 `~/.ssh/...` 절대 경로로 지정해야 함
- `Are you sure you want to continue connecting?` 질문에서 엔터만 누르면 기본 취소되어 `Host key verification failed` 오류가 남
- `-o StrictHostKeyChecking=accept-new` 옵션을 주면 질문 없이 새 서버의 호스트 키를 `~/.ssh/known_hosts`에 자동 등록하고 바로 연결됨
- 환경 설정 반영 시 점과 슬래시 순서에 주의: `~./bashrc` (X) -> `source ~/.bashrc` (O)

## 4. Hugging Face CLI 및 모델 캐시 확인

Hugging Face Hub의 모델을 내려받기 위한 CLI 설정과 다운로드된 모델 캐시 관리

LLM 가중치를 다운로드할 때 토큰 인증을 거치고, 내려받은 모델의 디스크 용량과 목록을 한눈에 파악하려고 씀

1. `pip install huggingface_hub` : Hugging Face Hub CLI 패키지 설치
2. `hf auth login` : Hugging Face Access Token 등록
3. `hf auth whoami` : 로그인 사용자 확인
4. `hf cache ls` : 다운로드된 허깅페이스 모델 캐시 목록 및 디스크 용량 확인
5. `cd /root/.cache/huggingface/hub/ && ls -a` : 캐시가 실제로 저장되는 디렉터리 경로 직접 확인

- `hf cache ls` : 허깅페이스 CLI에서 지원하는 캐시 스캔 명령으로, 저장소 이름(Repo ID), 종류(Model/Dataset), 디스크 사용량(Size on Disk), 최종 수정 시각을 표 형태로 깔끔하게 출력해 줌
- `hf cache rm` : 불필요하거나 용량을 많이 차지하는 모델 캐시를 대화형 메뉴로 골라서 안전하게 삭제할 때 사용
- 모델 가중치가 실제로 저장되는 경로는 `/root/.cache/huggingface/hub/` 아래이고, `models--[조직]--[모델명]` 형태의 폴더로 저장됨
- 내부에 실제 가중치 파일인 `blobs`와 특정 버전을 가리키는 `snapshots` 심볼릭 링크 구조로 관리됨
- 로컬(Windows)에서 로그인한 정보는 로컬에만 남으므로, 원격 서버에서 모델을 내려받으려면 원격 서버에서도 똑같이 로그인을 진행해야 함

## 5. Furiosa-LLM 및 NPU 다양한 모델 서빙 테스트

Furiosa NPU 가속기에서 모델 크기(Billion)와 종류를 바꿔가며 LLM을 서빙하고 실행하는 방법

모델 파라미터 크기(8B, 20B, 32B)와 용도에 따라 적합한 모델을 NPU에 올려 추론 성능과 메모리 점유율을 테스트하려고 씀

1. `furiosa-llm serve furiosa-ai/Qwen3-8B-FP8` : Qwen3 8B 경량 고속 모델 서빙
2. `furiosa-llm serve furiosa-ai/gpt-oss-20b` : GPT-OSS 20B 중형 모델 서빙
3. `furiosa-llm serve LGAI-EXAONE/EXAONE-4.0-32B-FP8` : EXAONE 4.0 32B 대형 모델 서빙
4. `furiosa-llm serve furiosa-ai/Meta-Llama-3.1-8B-Instruct-FP8-dynamic` : Llama 3.1 8B FP8 동적 양자화 모델 서빙
5. `fxb build Qwen/Qwen3-8B-FP8 qwen3-8b-fp8` : 허깅페이스 모델을 NPU 전용 실행 파일(`.fxb`)로 사전 컴파일

- 모델 이름의 `B`는 Billion(10억)이고, 신경망 가중치의 총 개수를 나타냄
- `8B`는 가벼워서 토큰 생성이 빠르고 NPU 메모리를 적게 차지함. `20B`·`32B`는 한국어 이해도와 복잡한 문맥 파악·추론 능력이 올라가는 대신 가중치와 KV 캐시를 담을 메모리가 더 필요함
- `fxb build`는 모델의 그래프 구조와 가중치를 NPU 전용 기계어로 컴파일해 단일 실행 파일로 묶음. C를 컴파일해 `.exe`를 만드는 것과 같은 자리
- 사전 컴파일해 두면 서빙 기동 때 컴파일을 기다리지 않고, `.fxb` 파일 하나로 다른 NPU 서버에 배포할 수 있음
- 서빙 중에 다른 터미널에서 `watch -n 1 furiosa-smi status`를 띄우면 모델 크기에 따라 메모리와 코어 사용률이 어떻게 달라지는지 봄
- `HTTP status client error (404 Not Found)`는 그 이름의 저장소나 태그(`revision`)가 없을 때 남. NPU용으로 사전 양자화해 올려둔 정확한 저장소 이름을 적어야 함

#### cf) 파라미터 개수와 자료형으로 메모리 계산

![8B 가중치 80억 개에 FP16(2바이트)을 곱하면 약 16GB, FP8(1바이트)을 곱하면 약 8GB가 되어 메모리가 절반으로 줄어든다](assets/params-bits-memory.svg)

- `8B`는 가중치의 개수(80억 개), `FP8`은 가중치 하나를 담는 그릇의 크기(1바이트)
- 가중치 메모리는 개수 × 자료형 크기. `80억 × 1바이트 = 약 8GB`이고, FP16(2바이트)이면 약 16GB
- 양자화는 16비트였던 가중치를 정밀도 손실을 줄이면서 8비트로 줄여 담는 것
- 용량이 절반이 되면 NPU가 메모리에서 가중치를 퍼 올리는 대역폭 병목이 줄고, FP8 전용 연산기를 쓸 수 있어 토큰 생성 속도가 올라감

#### cf) CUDA와 NPU 호환성의 비밀 & 사용률 96%의 의미

- NVIDIA CUDA는 엔비디아 GPU 전용 독점 기술이므로 NPU와 하드웨어나 메모리를 직접 공유하지 않음
- 기존 코드가 그대로 동작하는 이유는 Furiosa-LLM 런타임과 컴파일러가 PyTorch/vLLM 표준 인터페이스를 지원하여, 사용자가 짠 기존 코드를 NPU 전용 명령어로 실시간 변환해 실행해 주기 때문
- `furiosa-smi`의 사용률이 96%를 넘는 것은, CPU와 달리 AI 가속기(NPU)의 높은 점유율은 과부하가 아니라 연산 코어(PE)가 유휴(Idle) 시간 없이 풀파워로 텐서 연산을 처리하고 있다는 증거이며, 이 덕분에 빠른 토큰 추론 속도를 냄

## 6. 개발 환경 가이드: WSL vs SSH 언제 무엇을 쓸까?

로컬 리눅스 가상화(WSL)와 원격 서버 접속(SSH)의 역할 및 상황별 선택 기준

| 구분 | WSL (Windows Subsystem for Linux) | SSH (Secure Shell) |
| :--- | :--- | :--- |
| **컴퓨터 위치** | **내 컴퓨터 (로컬)** 안에서 돌아감 | **멀리 있는 서버실/클라우드 (원격)** 컴퓨터에 연결 |
| **사용 자원** | 내 노트북/데스크탑의 CPU, RAM, 로컬 디스크, 내장 GPU | 고성능 서버의 NPU, GPU, 대용량 RAM |
| **인터넷 연결** | 인터넷이 끊겨도 오프라인에서 자유롭게 작동 | 네트워크(인터넷) 연결이 반드시 유지되어야 함 |
| **윈도우 연동** | 윈도우 파일(`C:\`)과 실시간 파일 공유 및 상호 복사 가능 | 원격지이므로 `scp`나 `rsync`, git, VS Code Remote를 통해 전송 |
| **주요 용도** | • 가벼운 코드 작성, 스크립트 테스트, git 관리<br>• 리눅스 기반 개발 도구(Docker, bash) 로컬 실행<br>• 원격 서버 접속을 위한 베이스캠프 | • 초거대 모델(LLM) 파인튜닝, 대규모 학습<br>• 고성능 GPU/NPU 전용 프레임워크(furiosa-llm 등) 구동<br>• 24시간 중단 없이 가동되는 상용 서빙 API 배포 |

- 내 PC는 화면과 입력을 담당하고, WSL은 그 안의 리눅스 작업 환경이며, SSH는 무거운 연산을 서버에서 돌리기 위한 연결 통로